<a href="https://colab.research.google.com/github/rudalshan0412-code/attention-is-all-you-need-pytorch/blob/main/10)_Transformer_%EC%A1%B0%EB%A6%BD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Drive 연결

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 프로젝트 경로 설정

from pathlib import Path
import sys

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/attention_is_all_you_need"
)

SRC_DIR = PROJECT_ROOT / "src"

PROJECT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SRC_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)

PROJECT_ROOT: /content/drive/MyDrive/attention_is_all_you_need
SRC_DIR: /content/drive/MyDrive/attention_is_all_you_need/src


In [ ]:
# 현재 파일 구조 확인

for path in sorted(SRC_DIR.iterdir()):
    print(path.name)

__pycache__
attention.py
decoder.py
decoder_layer.py
encoder.py
encoder_layer.py
feed_forward.py
mask.py
multi_head_attention.py
positional_encoding.py


In [ ]:
# 기존 positional_encoding.py 확인

print(
    (SRC_DIR / "positional_encoding.py").read_text()
)


import math

import torch
import torch.nn as nn


class PositionalEncoding(nn.Module):
    """
    Sinusoidal Positional Encoding

    Input:
        (batch_size, seq_len, d_model)

    Output:
        (batch_size, seq_len, d_model)
    """

    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()

        if d_model % 2 != 0:
            raise ValueError("현재 구현에서는 d_model이 짝수여야 합니다.")

        self.d_model = d_model
        self.max_len = max_len

        self.dropout = nn.Dropout(dropout)

        position = torch.arange(
            max_len,
            dtype=torch.float32,
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32,
            )
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(
            max_len,
            d_model,
        )

        pe[:, 0::2] = torch.sin(
            position * 

In [ ]:
'''
sorce: encoder에 들어가는 input
target: decoder에 들어가는 input

target관련 추가 설명:

1) decoder에는 encoder의 최종 K와 V가 들어감
2) decoder에는 target 또한 들어감
3) 모든 토큰에 대해서 병렬적으로 진행하고 싶기에 target을 위치에 맞게 masking 해서 들어감.
'''

In [ ]:
# 기존 Encoder / Decoder / Mask 확인

print(
    (SRC_DIR / "encoder.py").read_text()
)
print(
    (SRC_DIR / "decoder.py").read_text()
)
print(
    (SRC_DIR / "mask.py").read_text()
)


import torch.nn as nn

from src.encoder_layer import EncoderLayer


class Encoder(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        num_layers,
        dropout=0.1,
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            EncoderLayer(
                d_model,
                num_heads,
                d_ff,
                dropout,
            )
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):
        attention_weights_list = []

        for layer in self.layers:
            x, attention_weights = layer(
                x,
                mask,
            )

            attention_weights_list.append(
                attention_weights
            )

        return x, attention_weights_list


import torch.nn as nn

from src.decoder_layer import DecoderLayer


class Decoder(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
  

In [ ]:
# src/transformer.py 전체 코드

import math

import torch.nn as nn

from src.positional_encoding import PositionalEncoding
from src.encoder import Encoder
from src.decoder import Decoder


class Transformer(nn.Module):
    def __init__(
        self,
        source_vocab_size,
        target_vocab_size,
        d_model,
        num_heads,
        d_ff,
        num_encoder_layers,
        num_decoder_layers,
        max_len,
        dropout=0.1,
    ):
        super().__init__()

        self.d_model = d_model

        self.source_embedding = nn.Embedding( # 번역 문제에서 source vocabulary와 target vocabulary가 다를 수 있기에 embedding을 따로 만든다.
            source_vocab_size,
            d_model,
        )

        self.target_embedding = nn.Embedding(
            target_vocab_size,
            d_model,
        )

        self.source_positional_encoding = PositionalEncoding(
            d_model,
            max_len,
            dropout,
        )

        self.target_positional_encoding = PositionalEncoding(
            d_model,
            max_len,
            dropout,
        )

        self.encoder = Encoder(
            d_model,
            num_heads,
            d_ff,
            num_encoder_layers,
            dropout,
        )

        self.decoder = Decoder(
            d_model,
            num_heads,
            d_ff,
            num_decoder_layers,
            dropout,
        )

        self.output_projection = nn.Linear(
            d_model,
            target_vocab_size,
        )

    def forward(
        self,
        source_token_ids,
        target_token_ids,
        source_mask=None,
        target_mask=None,
    ):
        source_x = self.source_embedding( # shape = (B, S) -> (B, S, d_model)   (ids -> embedding)
            source_token_ids
        )

        source_x = ( # shape 변화 없음(단순 제곱근 곱해준거임)
            source_x
            * math.sqrt(self.d_model) # embedding 직후 너무 크기가 너무 작기에 위치 정보에 휩쓸리지 않도록 균형을 맞추기 위해 sqrt(self.d_model)을 곱해줌
        )

        source_x = self.source_positional_encoding( # shape 변화 없음
            source_x
        )

        (
            encoder_output, # shape = (B, S, d_model)
            encoder_attention_weights_list, # shape = (B, H, Q, K)
        ) = self.encoder(
            source_x,
            source_mask,
        )

        target_x = self.target_embedding( # shape = (B, S) -> (B, S, d_model)
            target_token_ids
        )

        target_x = ( # shape 유지
            target_x
            * math.sqrt(self.d_model) # target 또한 마찬가지로 sqrt(d_model)을 곱해준다.
        )

        target_x = self.target_positional_encoding( # shape 유지
            target_x
        )

        (
            decoder_output, # (B, S, d_model)
            decoder_self_attention_weights_list,# shape = (B, H, Q_target, K_target)
            decoder_cross_attention_weights_list, # shape = (B, H, Q_target, K_source)
        ) = self.decoder(
            target_x,
            encoder_output,
            target_mask, # target mask는 target padding mask & casual mask이다. 따라서 target PAD key 차단과 미래의 token을 차단하는 역할을 한다.
            source_mask, # source와 같은 mask를 사용하는 이유: 둘 다 source PAD key를 보지 않아야하기에 동일한 mask를 재사용한다.
        )

        logits = self.output_projection( # shape = (B, Q, linear의 값)
            decoder_output
        )

        return (
            logits,
            encoder_attention_weights_list,
            decoder_self_attention_weights_list,
            decoder_cross_attention_weights_list,
        )

In [ ]:
# 코드 저장

%%writefile /content/drive/MyDrive/attention_is_all_you_need/src/transformer.py

import math

import torch.nn as nn

from src.positional_encoding import PositionalEncoding
from src.encoder import Encoder
from src.decoder import Decoder


class Transformer(nn.Module):
    def __init__(
        self,
        source_vocab_size,
        target_vocab_size,
        d_model,
        num_heads,
        d_ff,
        num_encoder_layers,
        num_decoder_layers,
        max_len,
        dropout=0.1,
    ):
        super().__init__()

        self.d_model = d_model

        self.source_embedding = nn.Embedding(
            source_vocab_size,
            d_model,
        )

        self.target_embedding = nn.Embedding(
            target_vocab_size,
            d_model,
        )

        self.source_positional_encoding = PositionalEncoding(
            d_model,
            max_len,
            dropout,
        )

        self.target_positional_encoding = PositionalEncoding(
            d_model,
            max_len,
            dropout,
        )

        self.encoder = Encoder(
            d_model,
            num_heads,
            d_ff,
            num_encoder_layers,
            dropout,
        )

        self.decoder = Decoder(
            d_model,
            num_heads,
            d_ff,
            num_decoder_layers,
            dropout,
        )

        self.output_projection = nn.Linear(
            d_model,
            target_vocab_size,
        )

    def forward(
        self,
        source_token_ids,
        target_token_ids,
        source_mask=None,
        target_mask=None,
    ):
        source_x = self.source_embedding(
            source_token_ids
        )

        source_x = (
            source_x
            * math.sqrt(self.d_model)
        )

        source_x = self.source_positional_encoding(
            source_x
        )

        (
            encoder_output,
            encoder_attention_weights_list,
        ) = self.encoder(
            source_x,
            source_mask,
        )

        target_x = self.target_embedding(
            target_token_ids
        )

        target_x = (
            target_x
            * math.sqrt(self.d_model)
        )

        target_x = self.target_positional_encoding(
            target_x
        )

        (
            decoder_output,
            decoder_self_attention_weights_list,
            decoder_cross_attention_weights_list,
        ) = self.decoder(
            target_x,
            encoder_output,
            target_mask,
            source_mask,
        )

        logits = self.output_projection(
            decoder_output
        )

        return (
            logits,
            encoder_attention_weights_list,
            decoder_self_attention_weights_list,
            decoder_cross_attention_weights_list,
        )

Writing /content/drive/MyDrive/attention_is_all_you_need/src/transformer.py


In [ ]:
# Transformer import

from src.transformer import Transformer

print("Transformer import 성공")

Transformer import 성공


In [ ]:
# 기본 Transformer 생성

source_vocab_size = 100
target_vocab_size = 120

d_model = 8
num_heads = 2
d_ff = 32

num_encoder_layers = 2
num_decoder_layers = 3

max_len = 50

dropout = 0.0

transformer = Transformer(
    source_vocab_size=source_vocab_size,
    target_vocab_size=target_vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_encoder_layers=num_encoder_layers,
    num_decoder_layers=num_decoder_layers,
    max_len=max_len,
    dropout=dropout,
)

print(transformer)

Transformer(
  (source_embedding): Embedding(100, 8)
  (target_embedding): Embedding(120, 8)
  (source_positional_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (target_positional_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Encoder(
    (layers): ModuleList(
      (0-1): 2 x EncoderLayer(
        (self_attention): MultiHeadAttention(
          (W_Q): Linear(in_features=8, out_features=8, bias=True)
          (W_K): Linear(in_features=8, out_features=8, bias=True)
          (W_V): Linear(in_features=8, out_features=8, bias=True)
          (attention): ScaledDotProductAttention()
          (W_O): Linear(in_features=8, out_features=8, bias=True)
        )
        (feed_forward): PositionwiseFeedForward(
          (linear1): Linear(in_features=8, out_features=32, bias=True)
          (linear2): Linear(in_features=32, out_features=8, bias=True)
        )
        (dropout1): Dropout(p=0.0, inplace=False)
    

In [ ]:
# 전체 Parameter 수

num_parameters = sum(
    p.numel()
    for p in transformer.parameters()
)

print("전체 Parameter 수:", num_parameters)

전체 Parameter 수: 8112


In [ ]:
# Embedding 객체 독립성 확인

print(
    transformer.source_embedding
    is transformer.target_embedding
)

False


In [ ]:
# Embedding Parameter 독립성 확인

print(
    transformer.source_embedding.weight
    is transformer.target_embedding.weight
)

False


In [ ]:
# Encoder와 Decoder 확인

from src.encoder import Encoder
from src.decoder import Decoder

print(
    isinstance(
        transformer.encoder,
        Encoder,
    )
)

print(
    isinstance(
        transformer.decoder,
        Decoder,
    )
)

True
True


In [ ]:
# Token IDs 생성

import torch

source_token_ids = torch.tensor([
    [10, 11, 12, 13, 14],
    [20, 21, 22, 0, 0],
])

target_token_ids = torch.tensor([
    [5, 8, 3, 9],
    [7, 2, 0, 0],
])

print(
    "source_token_ids: ",
    source_token_ids.shape,
    "target_token_ids: ",
    target_token_ids.shape,

)

source_token_ids:  torch.Size([2, 5]) target_token_ids:  torch.Size([2, 4])


In [ ]:
# 중간 Embedding Shape 직접 확인

source_embedding = (
    transformer.source_embedding(
        source_token_ids
    )
)

target_embedding = (
    transformer.target_embedding(
        target_token_ids
    )
)

print(
    "Source Embedding:",
    source_embedding.shape,
)

print(
    "Target Embedding:",
    target_embedding.shape,
)

Source Embedding: torch.Size([2, 5, 8])
Target Embedding: torch.Size([2, 4, 8])


In [ ]:
# Scaling Shape 확인

import math

source_scaled = (
    source_embedding
    * math.sqrt(d_model)
)

target_scaled = (
    target_embedding
    * math.sqrt(d_model)
)

print(source_scaled.shape)
print(target_scaled.shape)

torch.Size([2, 5, 8])
torch.Size([2, 4, 8])


In [ ]:
# Positional Encoding Shape 확인

source_x = (
    transformer.source_positional_encoding(
        source_scaled
    )
)

target_x = (
    transformer.target_positional_encoding(
        target_scaled
    )
)

print(
    "Source after PE:",
    source_x.shape,
)

print(
    "Target after PE:",
    target_x.shape,
)

Source after PE: torch.Size([2, 5, 8])
Target after PE: torch.Size([2, 4, 8])


In [ ]:
# Mask 없이 전체 Transformer 실행

(
    logits,
    encoder_weights,
    decoder_self_weights,
    decoder_cross_weights,
) = transformer(
    source_token_ids,
    target_token_ids,
)

In [ ]:
# 기본 출력 shape 확인

print(
    "logits:",
    logits.shape,
)

print(
    "encoder layers:",
    len(encoder_weights),
)

print(
    "decoder self layers:",
    len(decoder_self_weights),
)

print(
    "decoder cross layers:",
    len(decoder_cross_weights),
)

logits: torch.Size([2, 4, 120])
encoder layers: 2
decoder self layers: 3
decoder cross layers: 3


In [ ]:
# 모든 Attention Shape 확인


# Encoder
for i, weights in enumerate(
    encoder_weights
):
    print(
        f"Encoder Layer {i}:",
        weights.shape,
    )
# decoder self-attention

for i, weights in enumerate(
    decoder_self_weights
):
    print(
        f"Decoder Self Layer {i}:",
        weights.shape,
    )

# Cross-Attention

for i, weights in enumerate(
    decoder_cross_weights
):
    print(
        f"Decoder Cross Layer {i}:",
        weights.shape,
    )

Encoder Layer 0: torch.Size([2, 2, 5, 5])
Encoder Layer 1: torch.Size([2, 2, 5, 5])
Decoder Self Layer 0: torch.Size([2, 2, 4, 4])
Decoder Self Layer 1: torch.Size([2, 2, 4, 4])
Decoder Self Layer 2: torch.Size([2, 2, 4, 4])
Decoder Cross Layer 0: torch.Size([2, 2, 4, 5])
Decoder Cross Layer 1: torch.Size([2, 2, 4, 5])
Decoder Cross Layer 2: torch.Size([2, 2, 4, 5])


In [ ]:
# Mask 생성

from src.mask import (
    create_padding_mask,
    create_causal_mask,
)

pad_idx = 0

# source

source_mask = create_padding_mask(
    source_token_ids,
    pad_idx,
)

# Target padding

target_padding_mask = create_padding_mask(
    target_token_ids,
    pad_idx,
)

# Casual

target_len = target_token_ids.size(1)

causal_mask = create_causal_mask(
    target_len,
    device=target_token_ids.device,
)

# 결합

target_mask = (
    target_padding_mask
    & causal_mask
)

# shape 확인

print(
    "source_mask:",
    source_mask.shape,
)

print(
    "target_padding_mask:",
    target_padding_mask.shape,
)

print(
    "causal_mask:",
    causal_mask.shape,
)

print(
    "target_mask:",
    target_mask.shape,
)

source_mask: torch.Size([2, 1, 1, 5])
target_padding_mask: torch.Size([2, 1, 1, 4])
causal_mask: torch.Size([1, 1, 4, 4])
target_mask: torch.Size([2, 1, 4, 4])


In [ ]:
# 실제 Mask 값 확인

# Source Mask

print(
    "Source Mask:"
)

print(
    source_mask[:, 0, 0, :]
)

# target Mask

print(
    "Target Mask - Batch 0:"
)

print(
    target_mask[0, 0]
)

print(
    "\nTarget Mask - Batch 1:"
)

print(
    target_mask[1, 0]
)

Source Mask:
tensor([[ True,  True,  True,  True,  True],
        [ True,  True,  True, False, False]])
Target Mask - Batch 0:
tensor([[ True, False, False, False],
        [ True,  True, False, False],
        [ True,  True,  True, False],
        [ True,  True,  True,  True]])

Target Mask - Batch 1:
tensor([[ True, False, False, False],
        [ True,  True, False, False],
        [ True,  True, False, False],
        [ True,  True, False, False]])


In [ ]:
# Mask를 사용한 Transformer forward

(
    masked_logits,
    masked_encoder_weights,
    masked_decoder_self_weights,
    masked_decoder_cross_weights,
) = transformer(
    source_token_ids,
    target_token_ids,
    source_mask,
    target_mask,
)

# shape

print(masked_logits.shape)

torch.Size([2, 4, 120])


In [ ]:
# Encoder에서 Source PAD 차단 확인

# 두번째 batch의 source index 3 4 가 PAD이다

for layer_idx, weights in enumerate(
    masked_encoder_weights
):
    print(
        f"Encoder Layer {layer_idx}"
    )

    print(
        weights[
            1,
            :,
            :,
            3:
        ]
    )

Encoder Layer 0
tensor([[[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]]], grad_fn=<SliceBackward0>)
Encoder Layer 1
tensor([[[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]]], grad_fn=<SliceBackward0>)


In [ ]:
# Decoder Self-Attention 미래 token 차단

# Casual Mask 때문에 상삼각 부분은 모두 0이여야함

future_positions = torch.triu(
    torch.ones(
        target_len,
        target_len,
        dtype=torch.bool,
    ),
    diagonal=1,
)

# 검사

for weights in masked_decoder_self_weights:
    blocked_future = weights[
        :,
        :,
        future_positions,
    ]

    assert torch.allclose(
        blocked_future,
        torch.zeros_like(
            blocked_future
        ),
    )

print(
    "모든 Decoder Layer에서 "
    "미래 target key 차단 성공"
)

모든 Decoder Layer에서 미래 target key 차단 성공


In [ ]:
# Decoder Self-Attention Target PAD 차단

# 두번째 target이 [7, 2, PAD, PAD]이기에 key index 2, 3은 차단되어야한다.

for weights in masked_decoder_self_weights:
    pad_weights = weights[
        1,
        :,
        :,
        2:
    ]

    assert torch.allclose(
        pad_weights,
        torch.zeros_like(
            pad_weights
        ),
    )

print(
    "모든 Decoder Layer에서 "
    "target PAD key 차단 성공"
)

모든 Decoder Layer에서 target PAD key 차단 성공


In [ ]:
# Decoder Cross-Attention Source PAD 차단

# Cross-Attention에서도 동일한 source index가 차단되어야한다.

for weights in masked_decoder_self_weights:
    pad_weights = weights[
        1,
        :,
        :,
        2:
    ]

    assert torch.allclose(
        pad_weights,
        torch.zeros_like(
            pad_weights
        ),
    )

print(
    "모든 Decoder Layer에서 "
    "target PAD key 차단 성공"
)

모든 Decoder Layer에서 target PAD key 차단 성공


In [ ]:
# source_len != target_len 확인

# 현재 S = 5, T = 4 이다.

print(
    "source_len:",
    source_token_ids.size(1),
)

print(
    "target_len:",
    target_token_ids.size(1),
)

print(
    "Encoder Attention:",
    masked_encoder_weights[0].shape,
)

print(
    "Decoder Self:",
    masked_decoder_self_weights[0].shape,
)

print(
    "Decoder Cross:",
    masked_decoder_cross_weights[0].shape,
)

print(
    "Logits:",
    masked_logits.shape,
)

source_len: 5
target_len: 4
Encoder Attention: torch.Size([2, 2, 5, 5])
Decoder Self: torch.Size([2, 2, 4, 4])
Decoder Cross: torch.Size([2, 2, 4, 5])
Logits: torch.Size([2, 4, 120])


In [ ]:
# Batch / Sequence Length 변화 테스트

test_shapes = [
    (1, 5, 3),
    (2, 6, 4),
    (3, 4, 5),
]

for B, S, T in test_shapes:
    source_ids = torch.randint(
        0,
        source_vocab_size,
        (B, S),
    )

    target_ids = torch.randint(
        0,
        target_vocab_size,
        (B, T),
    )

    (
        test_logits,
        test_encoder_weights,
        test_decoder_self_weights,
        test_decoder_cross_weights,
    ) = transformer(
        source_ids,
        target_ids,
    )

    assert test_logits.shape == (
        B,
        T,
        target_vocab_size,
    )

    for weights in test_encoder_weights:
        assert weights.shape == (
            B,
            num_heads,
            S,
            S,
        )

    for weights in test_decoder_self_weights:
        assert weights.shape == (
            B,
            num_heads,
            T,
            T,
        )

    for weights in test_decoder_cross_weights:
        assert weights.shape == (
            B,
            num_heads,
            T,
            S,
        )

    print(
        f"B={B}, S={S}, T={T} 통과"
    )

B=1, S=5, T=3 통과
B=2, S=6, T=4 통과
B=3, S=4, T=5 통과


In [ ]:
# Encoder / Decoder Layer 수 변경 테스트

layer_cases = [
    (1, 1),
    (2, 3),
]

for enc_layers, dec_layers in layer_cases:
    test_transformer = Transformer(
        source_vocab_size,
        target_vocab_size,
        d_model,
        num_heads,
        d_ff,
        enc_layers,
        dec_layers,
        max_len,
        dropout=0.0,
    )

    (
        test_logits,
        test_encoder_weights,
        test_decoder_self_weights,
        test_decoder_cross_weights,
    ) = test_transformer(
        source_token_ids,
        target_token_ids,
    )

    assert len(
        test_transformer.encoder.layers
    ) == enc_layers

    assert len(
        test_transformer.decoder.layers
    ) == dec_layers

    assert len(
        test_encoder_weights
    ) == enc_layers

    assert len(
        test_decoder_self_weights
    ) == dec_layers

    assert len(
        test_decoder_cross_weights
    ) == dec_layers

    print(
        f"Encoder={enc_layers}, "
        f"Decoder={dec_layers} 통과"
    )

Encoder=1, Decoder=1 통과
Encoder=2, Decoder=3 통과


In [ ]:
# dropout = 0.0 반복 실행 동일성

# 현재 dropout = 0.0으로 생성했기에 같은 입력을 두번 실행한다

result1 = transformer(
    source_token_ids,
    target_token_ids,
    source_mask,
    target_mask,
)

result2 = transformer(
    source_token_ids,
    target_token_ids,
    source_mask,
    target_mask,
)

# Logits

assert torch.allclose(
    result1[0],
    result2[0],
)

# Encoder weights

for w1, w2 in zip(
    result1[1],
    result2[1],
):
    assert torch.allclose(
        w1,
        w2,
    )

# Decoder Self

for w1, w2 in zip(
    result1[2],
    result2[2],
):
    assert torch.allclose(
        w1,
        w2,
    )

# Decoder Cross

for w1, w2 in zip(
    result1[3],
    result2[3],
):
    assert torch.allclose(
        w1,
        w2,
    )

print(
    "dropout=0.0 반복 실행 동일성 확인 완료"
)

dropout=0.0 반복 실행 동일성 확인 완료


In [ ]:
# 최종 통합 테스트

(
    logits,
    encoder_weights,
    decoder_self_weights,
    decoder_cross_weights,
) = transformer(
    source_token_ids,
    target_token_ids,
    source_mask,
    target_mask,
)

B = source_token_ids.size(0)
S = source_token_ids.size(1)
T = target_token_ids.size(1)

# Embedding 독립성
assert (
    transformer.source_embedding
    is not transformer.target_embedding
)

assert (
    transformer.source_embedding.weight
    is not transformer.target_embedding.weight
)

# Layer 수
assert len(
    transformer.encoder.layers
) == num_encoder_layers

assert len(
    transformer.decoder.layers
) == num_decoder_layers

# Logits Shape
assert logits.shape == (
    B,
    T,
    target_vocab_size,
)

# Attention list 길이
assert len(
    encoder_weights
) == num_encoder_layers

assert len(
    decoder_self_weights
) == num_decoder_layers

assert len(
    decoder_cross_weights
) == num_decoder_layers

# Attention Shape
for weights in encoder_weights:
    assert weights.shape == (
        B,
        num_heads,
        S,
        S,
    )

for weights in decoder_self_weights:
    assert weights.shape == (
        B,
        num_heads,
        T,
        T,
    )

for weights in decoder_cross_weights:
    assert weights.shape == (
        B,
        num_heads,
        T,
        S,
    )

# Encoder source PAD 차단
for weights in encoder_weights:
    assert torch.all(
        weights[
            1,
            :,
            :,
            3:
        ] == 0
    )

# Decoder target PAD 차단
for weights in decoder_self_weights:
    assert torch.all(
        weights[
            1,
            :,
            :,
            2:
        ] == 0
    )

# Decoder 미래 token 차단
future_mask = torch.triu(
    torch.ones(
        T,
        T,
        dtype=torch.bool,
    ),
    diagonal=1,
)

for weights in decoder_self_weights:
    assert torch.all(
        weights[
            :,
            :,
            future_mask,
        ] == 0
    )

# Cross-Attention source PAD 차단
for weights in decoder_cross_weights:
    assert torch.all(
        weights[
            1,
            :,
            :,
            3:
        ] == 0
    )

print(
    "Transformer 전체 통합 테스트 통과"
)

Transformer 전체 통합 테스트 통과
